<a href="https://colab.research.google.com/github/chanceCoderByPassion/DL_CV/blob/master/Intro_12_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder


In [9]:
iris = load_iris()
df = pd.DataFrame(iris.data,columns=iris.feature_names)
df['target'] = iris.target

In [10]:
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [11]:
train_df,test_df = train_test_split(df,test_size=0.2,random_state=42,stratify=df['target'])

In [14]:
X_train,y_train = train_df.drop(columns='target'),train_df['target']
X_test,y_test = test_df.drop(columns='target'),test_df['target']

In [15]:
scaler = StandardScaler()
scaler.fit_transform(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
print(y_test.iloc[1])

2


In [22]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test.to_numpy(),dtype=torch.long)

In [23]:
class IrisClassifier(nn.Module):
  def __init__(self,input_size,hidden_size,output_size):
    super(IrisClassifier,self).__init__()

    self.network = nn.Sequential(
        nn.Linear(input_size,hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size,hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size,output_size)
    )

  def forward(self,x):
    return self.network(x)


In [26]:
input_size = X_train.shape[1]
hidden_size = 16
output_size = 3

In [27]:
model = IrisClassifier(input_size,hidden_size,output_size)

In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [29]:
# Train the model
epochs = 500
for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()
  forward_pass = model(X_train_tensor)
  loss = criterion(forward_pass,y_train_tensor)
  loss.backward()
  optimizer.step()

  if (epoch+1) % 50 == 0:
    print(f"Epoch:{epoch+1}/{epochs},Loss:{loss.item():.4f}")

Epoch:50/500,Loss:0.8939
Epoch:100/500,Loss:0.5701
Epoch:150/500,Loss:0.2701
Epoch:200/500,Loss:0.1437
Epoch:250/500,Loss:0.0895
Epoch:300/500,Loss:0.0653
Epoch:350/500,Loss:0.0526
Epoch:400/500,Loss:0.0450
Epoch:450/500,Loss:0.0400
Epoch:500/500,Loss:0.0363


In [30]:
model.eval()
with torch.no_grad():
  y_pred = model(X_test_tensor)
  y_pred_labels = torch.argmax(y_pred,dim=1)

  accuracy = (y_pred_labels == y_test_tensor).sum().item() / (y_test_tensor.size(0))
  print(f"Test accuracy is:{accuracy:.4f}")

Test accuracy is:0.9667


In [37]:
def predict_iris(sepal_length,sepal_width,petal_length,petal_width):
  input_data=np.array([[sepal_length,sepal_width,petal_length,petal_width]])
  input_data = scaler.transform(input_data)
  input_tensor = torch.tensor(input_data,dtype=torch.float32)

  model.eval()
  with torch.no_grad():
    y_pred = model(input_tensor)
    pred_class = torch.argmax(y_pred,dim=1).item()

  return iris.target_names[pred_class]

In [38]:
predicted_class = predict_iris(5.1,3.5,1.4,0.2)
print(f"Predicted species:{predicted_class}")

Predicted species:setosa


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
